In [59]:
# ============================================================
# MODELO CHURN MEJORADO REAL
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

from imblearn.over_sampling import SMOTE

# ============================================================
# VARIABLES
# ============================================================

variables = [

    "edad",
    "antiguedad_meses",
    "ingreso_estimado",
    "num_lineas",
    "descuento_activo",

    "importe_total",
    "dias_retraso_pago",
    "impago_flag",

    "importe_mes_anterior",
    "consumo_mes_anterior",
    "retraso_mes_anterior",
    "impago_mes_anterior",

    "media_importe_3m",
    "media_retraso_3m",
    "impagos_3m",

    "cambio_consumo",
    "subida_factura",
    "variacion_consumo_pct",

    "stress_calidad_lag",
    "incidencia_masiva_lag"

]

# ============================================================
# ELIMINAR NULLS
# ============================================================

df_model = df_model.dropna(subset=variables + ["churn"])

# ============================================================
# X / y
# ============================================================

X = df_model[variables]
y = df_model["churn"]

# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y

)

# ============================================================
# SMOTE SOLO TRAIN
# ============================================================

smote = SMOTE(
    sampling_strategy=0.50,
    random_state=42,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

# ============================================================
# DISTRIBUCIÓN
# ============================================================

print("="*60)
print("DISTRIBUCIÓN TRAIN DESPUÉS DE SMOTE")
print("="*60)

print(y_train_smote.value_counts())

# ============================================================
# MODELO RANDOM FOREST
# ============================================================

modelo = RandomForestClassifier(

    n_estimators=400,
    max_depth=10,
    min_samples_split=15,
    min_samples_leaf=8,

    class_weight="balanced",

    random_state=42,
    n_jobs=-1

)

# ============================================================
# ENTRENAMIENTO
# ============================================================

modelo.fit(X_train_smote, y_train_smote)

# ============================================================
# PROBABILIDADES
# ============================================================

y_prob = modelo.predict_proba(X_test)[:, 1]

# ============================================================
# THRESHOLD
# ============================================================

threshold = 0.35

y_pred = (y_prob >= threshold).astype(int)

# ============================================================
# MÉTRICAS
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

roc = roc_auc_score(y_test, y_prob)

# ============================================================
# RESULTADOS
# ============================================================

print("\n" + "="*60)
print("RESULTADOS MODELO FINAL")
print("="*60)

print(f"\nAccuracy: {accuracy:.2%}")
print(f"Precision churn: {precision:.4f}")
print(f"Recall churn: {recall:.4f}")
print(f"F1 churn: {f1:.4f}")
print(f"ROC-AUC: {roc:.4f}")

# ============================================================
# MATRIZ CONFUSIÓN
# ============================================================

print("\nMatriz de confusión:")

print(
    confusion_matrix(y_test, y_pred)
)

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:\n")

print(
    classification_report(y_test, y_pred)
)

# ============================================================
# IMPORTANCIA VARIABLES
# ============================================================

importancias = pd.DataFrame({

    "Variable": variables,
    "Importancia": modelo.feature_importances_

})

importancias = importancias.sort_values(
    by="Importancia",
    ascending=False
)

print("\n" + "="*60)
print("IMPORTANCIA VARIABLES")
print("="*60)

print(importancias.head(15))

DISTRIBUCIÓN TRAIN DESPUÉS DE SMOTE
churn
0    13669
1     6834
Name: count, dtype: int64

RESULTADOS MODELO FINAL

Accuracy: 67.06%
Precision churn: 0.0748
Recall churn: 0.3586
F1 churn: 0.1237
ROC-AUC: 0.5572

Matriz de confusión:
[[2366 1052]
 [ 152   85]]

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.69      0.80      3418
           1       0.07      0.36      0.12       237

    accuracy                           0.67      3655
   macro avg       0.51      0.53      0.46      3655
weighted avg       0.88      0.67      0.75      3655


IMPORTANCIA VARIABLES
                 Variable  Importancia
14             impagos_3m     0.169703
18     stress_calidad_lag     0.141644
13       media_retraso_3m     0.091286
4        descuento_activo     0.071345
17  variacion_consumo_pct     0.057433
11    impago_mes_anterior     0.056218
2        ingreso_estimado     0.055755
6       dias_retraso_pago     0.044867
9    consumo_m

In [60]:
# ============================================================
# MÉTRICAS
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

print("\n" + "="*60)
print("RESULTADOS MODELO")
print("="*60)

print("\nAccuracy:")
print(round(accuracy * 100, 2), "%")

print("\nPrecision:")
print(round(precision_score(y_test, y_pred, zero_division=0), 4))

print("\nRecall:")
print(round(recall_score(y_test, y_pred, zero_division=0), 4))

print("\nF1-score:")
print(round(f1_score(y_test, y_pred, zero_division=0), 4))

print("\nROC-AUC:")
print(round(roc_auc_score(y_test, y_prob), 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


RESULTADOS MODELO

Accuracy:
67.06 %

Precision:
0.0748

Recall:
0.3586

F1-score:
0.1237

ROC-AUC:
0.5572

Matriz de confusión:
[[2366 1052]
 [ 152   85]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.69      0.80      3418
           1       0.07      0.36      0.12       237

    accuracy                           0.67      3655
   macro avg       0.51      0.53      0.46      3655
weighted avg       0.88      0.67      0.75      3655

